In [8]:
#basics
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings("ignore")


#preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.base import BaseEstimator, TransformerMixin
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
from sklearn.manifold import TSNE


#transformers and pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


#algorithms
from sklearn.ensemble import GradientBoostingRegressor
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras import regularizers



#model evaluation
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import ShuffleSplit
from sklearn.model_selection import cross_val_score
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import optuna
from optuna.samplers import TPESampler
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_slice



# To store data
import pandas as pd


# To do linear algebra
import numpy as np


# To create plots
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches


# To create interactive plots
from plotly.offline import init_notebook_mode
import plotly.graph_objs as go
init_notebook_mode(connected=True)

# To build models
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA


### Import dataset

In [ ]:
original_train = pd.read_csv("Data/train.csv")
original_test = pd.read_csv("Data/test.csv")

train_df = original_train.copy()
test_df = original_test.copy()

# Dataset union
train_df = pd.concat([train_df, test_df])

### Covariates renaming

In [10]:
# Rename columns in-place so that they only have letters, digits, and underscores:
train_df.columns = [
    re.sub(r"[^A-Za-z0-9_]+", "_", col)  # Replace non-alphanumeric/underscore chars with "_"
    for col in train_df.columns
]

In [11]:
X = train_df.drop(columns="Activity")
y = train_df["Activity"]

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

In [12]:
# Covariates' name
features = [feature for feature in X.columns]

# GBM tuning

In [18]:
scaler_pipe = Pipeline(steps=[
    ("Scaler", StandardScaler())
])

scaler_pipe

Pipeline(steps=[('Scaler', StandardScaler())])

In [19]:
base_preprocess = ColumnTransformer(remainder="passthrough",
                                    transformers=[("Scaler", scaler_pipe, features)
                                                  ])
base_preprocess

ColumnTransformer(remainder='passthrough',
                  transformers=[('Scaler',
                                 Pipeline(steps=[('Scaler', StandardScaler())]),
                                 ['tBodyAcc_mean_X', 'tBodyAcc_mean_Y',
                                  'tBodyAcc_mean_Z', 'tBodyAcc_std_X',
                                  'tBodyAcc_std_Y', 'tBodyAcc_std_Z',
                                  'tBodyAcc_mad_X', 'tBodyAcc_mad_Y',
                                  'tBodyAcc_mad_Z', 'tBodyAcc_max_X',
                                  'tBodyAcc_max_Y', 'tBodyAcc_max_Z',
                                  'tBodyAcc_min_X', 'tBodyAcc_min_Y',
                                  'tBodyAcc_min_Z', 'tBodyAcc_sma_',
                                  'tBodyAcc_energy_X', 'tBodyAcc_energy_Y',
                                  'tBodyAcc_energy_Z', 'tBodyAcc_iqr_X',
                                  'tBodyAcc_iqr_Y', 'tBodyAcc_iqr_Z',
                                  'tBodyAcc_entropy_X', 'tBodyAcc_entropy_Y',
                                  'tBodyAcc_entropy_Z', 'tBodyAcc_arCoeff_X_1',
                                  'tBodyAcc_arCoeff_X_2',
                                  'tBodyAcc_arCoeff_X_3',
                                  'tBodyAcc_arCoeff_X_4',
                                  'tBodyAcc_arCoeff_Y_1', ...])])

In [ ]:
# import optuna
# from optuna.samplers import TPESampler
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import make_pipeline
# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.model_selection import ShuffleSplit, cross_val_score
# from sklearn.metrics import make_scorer, accuracy_score

# def objective(trial):
#     n_estimators = trial.suggest_int("n_estimators", 500, 1000, step=100)
#     min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10, step=1)
#     max_depth = trial.suggest_int("max_depth", 1, 8, step=2)
    
#     gbm_classifier = GradientBoostingClassifier(
#         n_estimators=n_estimators,
#         min_samples_leaf=min_samples_leaf,
#         max_depth=max_depth,
#         learning_rate=0.05,
#         subsample=0.6,
#         max_features="sqrt",
#         random_state=1
#     )
    
#     # Create the full pipeline with preprocessing and classifier
#     pipeline = make_pipeline(base_preprocess, gbm_classifier)
    
#     # Use ShuffleSplit for cross-validation
#     ss = ShuffleSplit(n_splits=10, test_size=0.3, random_state=0)
#     scores = cross_val_score(pipeline, X, y_encoded,
#                              scoring=make_scorer(accuracy_score),
#                              cv=ss)
#     return scores.mean()

# # Set up the study with TPESampler for reproducibility
# sampler = TPESampler(seed=1)
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(objective, n_trials=10)

[I 2025-03-24 10:37:07,733] A new study created in memory with name: no-name-48113a49-df14-4b16-a92c-5729db3fc52f
[I 2025-03-24 10:42:45,163] Trial 0 finished with value: 0.98084142394822 and parameters: {'n_estimators': 700, 'min_samples_leaf': 8, 'max_depth': 1}. Best is trial 0 with value: 0.98084142394822.
[I 2025-03-24 10:47:35,983] Trial 1 finished with value: 0.9780258899676376 and parameters: {'n_estimators': 600, 'min_samples_leaf': 2, 'max_depth': 1}. Best is trial 0 with value: 0.98084142394822.
[I 2025-03-24 11:01:08,813] Trial 2 finished with value: 0.9914239482200647 and parameters: {'n_estimators': 600, 'min_samples_leaf': 4, 'max_depth': 3}. Best is trial 2 with value: 0.9914239482200647.
[I 2025-03-24 11:25:22,831] Trial 3 finished with value: 0.9927184466019418 and parameters: {'n_estimators': 800, 'min_samples_leaf': 5, 'max_depth': 5}. Best is trial 3 with value: 0.9927184466019418.
[I 2025-03-24 11:30:13,635] Trial 4 finished with value: 0.9790291262135924 and para

In [ ]:
# plot_optimization_history(study)

In [ ]:
# plot_slice(study)

In [ ]:
# plot_param_importances(study)

In [ ]:
# [I 2025-03-24 10:37:07,733] A new study created in memory with name: no-name-48113a49-df14-4b16-a92c-5729db3fc52f
# [I 2025-03-24 10:42:45,163] Trial 0 finished with value: 0.98084142394822 and parameters: {'n_estimators': 700, 'min_samples_leaf': 8, 'max_depth': 1}. Best is trial 0 with value: 0.98084142394822.
# [I 2025-03-24 10:47:35,983] Trial 1 finished with value: 0.9780258899676376 and parameters: {'n_estimators': 600, 'min_samples_leaf': 2, 'max_depth': 1}. Best is trial 0 with value: 0.98084142394822.
# [I 2025-03-24 11:01:08,813] Trial 2 finished with value: 0.9914239482200647 and parameters: {'n_estimators': 600, 'min_samples_leaf': 4, 'max_depth': 3}. Best is trial 2 with value: 0.9914239482200647.
# [I 2025-03-24 11:25:22,831] Trial 3 finished with value: 0.9927184466019418 and parameters: {'n_estimators': 800, 'min_samples_leaf': 5, 'max_depth': 5}. Best is trial 3 with value: 0.9927184466019418.
# [I 2025-03-24 11:30:13,635] Trial 4 finished with value: 0.9790291262135924 and parameters: {'n_estimators': 600, 'min_samples_leaf': 9, 'max_depth': 1}. Best is trial 3 with value: 0.9927184466019418.
# [I 2025-03-24 11:55:49,166] Trial 5 finished with value: 0.9927831715210356 and parameters: {'n_estimators': 900, 'min_samples_leaf': 5, 'max_depth': 5}. Best is trial 5 with value: 0.9927831715210356.
# [I 2025-03-24 12:18:23,516] Trial 6 finished with value: 0.9914239482200647 and parameters: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_depth': 7}. Best is trial 5 with value: 0.9927831715210356.
# [I 2025-03-24 12:45:23,731] Trial 7 finished with value: 0.9931715210355987 and parameters: {'n_estimators': 1000, 'min_samples_leaf': 4, 'max_depth': 5}. Best is trial 7 with value: 0.9931715210355987.
# [I 2025-03-24 12:53:23,492] Trial 8 finished with value: 0.9848543689320388 and parameters: {'n_estimators': 1000, 'min_samples_leaf': 9, 'max_depth': 1}. Best is trial 7 with value: 0.9931715210355987.
# [I 2025-03-24 13:16:02,313] Trial 9 finished with value: 0.9914239482200647 and parameters: {'n_estimators': 500, 'min_samples_leaf': 2, 'max_depth': 7}. Best is trial 7 with value: 0.9931715210355987.
# 158m 54.6s

{'n_estimators': 1000, 'min_samples_leaf': 4, 'max_depth': 5}

# GBM

In [ ]:
base_boost = Pipeline(steps=[
    ("Scaler", base_preprocess),
    ("Boost", GradientBoostingClassifier(n_estimators=1000,
        min_samples_leaf=4,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.6,
        max_features="sqrt",
        random_state=1))
])

base_boost

Pipeline(steps=[('Scaler',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Scaler',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler())]),
                                                  ['tBodyAcc_mean_X',
                                                   'tBodyAcc_mean_Y',
                                                   'tBodyAcc_mean_Z',
                                                   'tBodyAcc_std_X',
                                                   'tBodyAcc_std_Y',
                                                   'tBodyAcc_std_Z',
                                                   'tBodyAcc_mad_X',
                                                   'tBodyAcc_mad_Y',
                                                   'tBodyAcc_mad_Z',
                                                   'tBodyAcc_max_X',
                                                   'tBodyAcc_max_Y',
                                                   'tBodyAcc_max_Z',
                                                   'tBo...
                                                   'tBodyAcc_iqr_Z',
                                                   'tBodyAcc_entropy_X',
                                                   'tBodyAcc_entropy_Y',
                                                   'tBodyAcc_entropy_Z',
                                                   'tBodyAcc_arCoeff_X_1',
                                                   'tBodyAcc_arCoeff_X_2',
                                                   'tBodyAcc_arCoeff_X_3',
                                                   'tBodyAcc_arCoeff_X_4',
                                                   'tBodyAcc_arCoeff_Y_1', ...])])),
                ('Tree',
                 GradientBoostingClassifier(learning_rate=0.05, max_depth=5,
                                            max_features='sqrt',
                                            min_samples_leaf=4,
                                            n_estimators=1000, random_state=1,
                                            subsample=0.6))])

### Fit the model

In [31]:
acc_scores = cross_val_score(base_boost, X, y_encoded, cv=10, scoring="accuracy")
f1_scores = cross_val_score(base_boost, X, y_encoded, cv=10, scoring="f1_macro")

print("Accuracy scores for each of the 10 folds:", acc_scores)
print("Mean accuracy:", acc_scores.mean())
print("F1 scores for each of the 10 folds:", f1_scores)
print("Mean accuracy:", f1_scores.mean())

Accuracy scores for each of the 10 folds: [0.96796117 0.94757282 0.87961165 0.97669903 0.97961165 0.98349515
 0.95825243 0.95242718 0.95048544 0.98056365]
Mean accuracy: 0.957668015888741
F1 scores for each of the 10 folds: [0.97006388 0.94458874 0.87856077 0.97750605 0.97877971 0.98337889
 0.96056118 0.95392666 0.95129343 0.97813279]
Mean accuracy: 0.957679209940418


In [32]:
boost_results = {"accuracy": acc_scores, "F1": f1_scores}

------------

# PCA+GBM

In [36]:
PCA_pipe = Pipeline(steps=[
    ("Scaler", StandardScaler()),
    ("pca", PCA(n_components=35))
])

PCA_pipe

Pipeline(steps=[('Scaler', StandardScaler()), ('pca', PCA(n_components=35))])

In [37]:
PCA_preprocess = ColumnTransformer(remainder="drop",
                                    transformers=[("PCA", PCA_pipe, features)
                                                  ])
PCA_preprocess

ColumnTransformer(transformers=[('PCA',
                                 Pipeline(steps=[('Scaler', StandardScaler()),
                                                 ('pca',
                                                  PCA(n_components=35))]),
                                 ['tBodyAcc_mean_X', 'tBodyAcc_mean_Y',
                                  'tBodyAcc_mean_Z', 'tBodyAcc_std_X',
                                  'tBodyAcc_std_Y', 'tBodyAcc_std_Z',
                                  'tBodyAcc_mad_X', 'tBodyAcc_mad_Y',
                                  'tBodyAcc_mad_Z', 'tBodyAcc_max_X',
                                  'tBodyAcc_max_Y', 'tBodyAcc_max_Z',
                                  'tBodyAcc_min_X', 'tBodyAcc_min_Y',
                                  'tBodyAcc_min_Z', 'tBodyAcc_sma_',
                                  'tBodyAcc_energy_X', 'tBodyAcc_energy_Y',
                                  'tBodyAcc_energy_Z', 'tBodyAcc_iqr_X',
                                  'tBodyAcc_iqr_Y', 'tBodyAcc_iqr_Z',
                                  'tBodyAcc_entropy_X', 'tBodyAcc_entropy_Y',
                                  'tBodyAcc_entropy_Z', 'tBodyAcc_arCoeff_X_1',
                                  'tBodyAcc_arCoeff_X_2',
                                  'tBodyAcc_arCoeff_X_3',
                                  'tBodyAcc_arCoeff_X_4',
                                  'tBodyAcc_arCoeff_Y_1', ...])])

In [39]:
PCA_boost = Pipeline(steps=[
    ("preprocess", PCA_preprocess),
    ("Boost", GradientBoostingClassifier(n_estimators=1000,
        min_samples_leaf=4,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.6,
        max_features="sqrt",
        random_state=1))
])

PCA_boost

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('PCA',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler()),
                                                                  ('pca',
                                                                   PCA(n_components=35))]),
                                                  ['tBodyAcc_mean_X',
                                                   'tBodyAcc_mean_Y',
                                                   'tBodyAcc_mean_Z',
                                                   'tBodyAcc_std_X',
                                                   'tBodyAcc_std_Y',
                                                   'tBodyAcc_std_Z',
                                                   'tBodyAcc_mad_X',
                                                   'tBodyAcc_mad_Y',
                                                   'tBodyAcc_mad_Z',
                                                   'tBodyAcc_max_X',
                                                   'tBodyAcc_max_Y',
                                                   'tBodyAcc_max_Z...
                                                   'tBodyAcc_iqr_Z',
                                                   'tBodyAcc_entropy_X',
                                                   'tBodyAcc_entropy_Y',
                                                   'tBodyAcc_entropy_Z',
                                                   'tBodyAcc_arCoeff_X_1',
                                                   'tBodyAcc_arCoeff_X_2',
                                                   'tBodyAcc_arCoeff_X_3',
                                                   'tBodyAcc_arCoeff_X_4',
                                                   'tBodyAcc_arCoeff_Y_1', ...])])),
                ('Boost',
                 GradientBoostingClassifier(learning_rate=0.05, max_depth=5,
                                            max_features='sqrt',
                                            min_samples_leaf=4,
                                            n_estimators=1000, random_state=1,
                                            subsample=0.6))])

In [41]:
acc_scores = cross_val_score(PCA_boost, X, y_encoded, cv=10, scoring="accuracy")
f1_scores = cross_val_score(PCA_boost, X, y_encoded, cv=10, scoring="f1_macro")

print("Accuracy scores for each of the 10 folds:", acc_scores)
print("Mean accuracy:", acc_scores.mean())
print("F1 scores for each of the 10 folds:", f1_scores)
print("Mean F1:", f1_scores.mean())

pca_results = {"accuracy": acc_scores, "F1": f1_scores}

Accuracy scores for each of the 10 folds: [0.93883495 0.89029126 0.83106796 0.89708738 0.90291262 0.93495146
 0.91165049 0.92912621 0.90679612 0.91836735]
Mean accuracy: 0.9061085793540717
F1 scores for each of the 10 folds: [0.94075604 0.88754828 0.82989068 0.89665075 0.90011346 0.93753707
 0.91521451 0.9328577  0.90867766 0.90859247]
Mean F1: 0.9057838607266671


----------------------------

# LDA+GBM

In [42]:
class LDATransformer(BaseEstimator, TransformerMixin):
    def __init__(self, n_components=None):
        self.n_components = n_components
        self.lda = None
    
    def fit(self, X, y):
        self.lda = LinearDiscriminantAnalysis(n_components=self.n_components)
        self.lda.fit(X, y)
        return self
    
    def transform(self, X):
        return self.lda.transform(X)
    
    def fit_transform(self, X, y):
        return self.fit(X, y).transform(X)

# Define the pipeline
LDA_pipe = Pipeline(steps=[
    ("Scaler", StandardScaler()), 
    ("LDA", LDATransformer(n_components=5))
])

LDA_pipe

Pipeline(steps=[('Scaler', StandardScaler()),
                ('LDA', LDATransformer(n_components=5))])

In [43]:
LDA_preprocess = ColumnTransformer(remainder="drop",
                                    transformers=[("LDA", LDA_pipe, list(range(X.shape[1])))
                                                  ])
LDA_preprocess

ColumnTransformer(transformers=[('LDA',
                                 Pipeline(steps=[('Scaler', StandardScaler()),
                                                 ('LDA',
                                                  LDATransformer(n_components=5))]),
                                 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                  14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24,
                                  25, 26, 27, 28, 29, ...])])

In [47]:
LDA_boost = Pipeline(steps=[
    ("LDA", LDA_preprocess),
    ("Tree", GradientBoostingClassifier(n_estimators=1000,
        min_samples_leaf=4,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.6,
        max_features="sqrt",
        random_state=1))
])

LDA_boost

Pipeline(steps=[('LDA',
                 ColumnTransformer(transformers=[('LDA',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler()),
                                                                  ('LDA',
                                                                   LDATransformer(n_components=5))]),
                                                  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                   10, 11, 12, 13, 14, 15, 16,
                                                   17, 18, 19, 20, 21, 22, 23,
                                                   24, 25, 26, 27, 28, 29, ...])])),
                ('Tree',
                 GradientBoostingClassifier(learning_rate=0.05, max_depth=5,
                                            max_features='sqrt',
                                            min_samples_leaf=4,
                                            n_estimators=1000, random_state=1,
                                            subsample=0.6))])

In [48]:
acc_scores = cross_val_score(LDA_boost, X, y_encoded, cv=10, scoring="accuracy")
f1_scores = cross_val_score(LDA_boost, X, y_encoded, cv=10, scoring="f1_macro")

print("Accuracy scores for each of the 10 folds:", acc_scores)
print("Mean accuracy:", acc_scores.mean())
print("F1 scores for each of the 10 folds:", f1_scores)
print("Mean F1:", f1_scores.mean())

LDA_results = {"accuracy": acc_scores, "F1": f1_scores}

Accuracy scores for each of the 10 folds: [0.96504854 0.93203883 0.89514563 0.95145631 0.97961165 0.97669903
 0.97669903 0.96699029 0.97572816 0.97084548]
Mean accuracy: 0.959026295677772
F1 scores for each of the 10 folds: [0.96679039 0.92989381 0.89358721 0.95113427 0.9791229  0.97825059
 0.9780321  0.96832631 0.97691435 0.97072923]
Mean F1: 0.9592781151564458


-----------

# LDA+GBM tuning

In [60]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 500, 1000, step=100)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10, step=1)
    max_depth = trial.suggest_int("max_depth", 1, 8, step=2)
    
    gbm_classifier = GradientBoostingClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        max_depth=max_depth,
        learning_rate=0.05,
        subsample=0.6,
        max_features="sqrt",
        random_state=1
    )
    
    # Create the full pipeline with preprocessing and classifier
    pipeline = make_pipeline(LDA_preprocess, gbm_classifier)
    
    # Use ShuffleSplit for cross-validation
    ss = ShuffleSplit(n_splits=10, test_size=0.3, random_state=0)
    scores = cross_val_score(pipeline, X, y_encoded,
                             scoring=make_scorer(accuracy_score),
                             cv=ss)
    return scores.mean()

# Set up the study with TPESampler for reproducibility
sampler = TPESampler(seed=1)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=10)

[I 2025-03-24 17:45:47,933] A new study created in memory with name: no-name-d9dabc31-e45e-47f7-a200-d6fa552e8259
[I 2025-03-24 17:46:49,853] Trial 0 finished with value: 0.9799352750809062 and parameters: {'n_estimators': 700, 'min_samples_leaf': 8, 'max_depth': 1}. Best is trial 0 with value: 0.9799352750809062.
[I 2025-03-24 17:47:43,827] Trial 1 finished with value: 0.980032362459547 and parameters: {'n_estimators': 600, 'min_samples_leaf': 2, 'max_depth': 1}. Best is trial 1 with value: 0.980032362459547.
[I 2025-03-24 17:49:29,145] Trial 2 finished with value: 0.9807443365695793 and parameters: {'n_estimators': 600, 'min_samples_leaf': 4, 'max_depth': 3}. Best is trial 2 with value: 0.9807443365695793.
[I 2025-03-24 17:52:09,180] Trial 3 finished with value: 0.9809708737864078 and parameters: {'n_estimators': 800, 'min_samples_leaf': 5, 'max_depth': 5}. Best is trial 3 with value: 0.9809708737864078.
[I 2025-03-24 17:53:04,437] Trial 4 finished with value: 0.9801618122977345 and 

In [61]:
# {'n_estimators': 500, 'min_samples_leaf': 2, 'max_depth': 7}
LDA_tuned = Pipeline(steps=[
    ("LDA", LDA_preprocess),
    ("Tree", GradientBoostingClassifier(n_estimators=500,
        min_samples_leaf=2,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.6,
        max_features="sqrt",
        random_state=1))
])

LDA_tuned

Pipeline(steps=[('LDA',
                 ColumnTransformer(transformers=[('LDA',
                                                  Pipeline(steps=[('Scaler',
                                                                   StandardScaler()),
                                                                  ('LDA',
                                                                   LDATransformer(n_components=5))]),
                                                  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                   10, 11, 12, 13, 14, 15, 16,
                                                   17, 18, 19, 20, 21, 22, 23,
                                                   24, 25, 26, 27, 28, 29, ...])])),
                ('Tree',
                 GradientBoostingClassifier(learning_rate=0.05, max_depth=7,
                                            max_features='sqrt',
                                            min_samples_leaf=2,
                                            n_estimators=500, random_state=1,
                                            subsample=0.6))])

In [62]:
acc_scores = cross_val_score(LDA_tuned, X, y_encoded, cv=10, scoring="accuracy")
f1_scores = cross_val_score(LDA_tuned, X, y_encoded, cv=10, scoring="f1_macro")

print("Accuracy scores for each of the 10 folds:", acc_scores)
print("Mean accuracy:", acc_scores.mean())
print("F1 scores for each of the 10 folds:", f1_scores)
print("Mean F1:", f1_scores.mean())

Tuned_results = {"accuracy": acc_scores, "F1": f1_scores}

Accuracy scores for each of the 10 folds: [0.96601942 0.93398058 0.89320388 0.95242718 0.98058252 0.97572816
 0.97572816 0.97184466 0.97669903 0.97278912]
Mean accuracy: 0.9599002707879268
F1 scores for each of the 10 folds: [0.96768905 0.93155576 0.89175963 0.95208632 0.98007528 0.97734688
 0.97712999 0.97313619 0.97782721 0.97244776]
Mean F1: 0.9601054067044557


----

# LdaBoost

In [ ]:
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeRegressor
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

X = X.to_numpy()

# ---------- utility ----------
def softmax(F):
    """Computes the softmax row-wise."""
    expF = np.exp(F - np.max(F, axis=1, keepdims=True))
    return expF / np.sum(expF, axis=1, keepdims=True)


# ---------- model ----------
class CustomMulticlassGradientBoostingClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

        # attributes that will store the learned state (with "_" suffix)
        self.estimators_ = None
        self.lda_transforms_ = None
        self.initial_logit_ = None
        self.classes_ = None

    # -------------------------- training --------------------------
    def fit(self, X, y):
        # reset the state
        self.estimators_ = []
        self.lda_transforms_ = []
        self.initial_logit_ = None
        self.classes_ = None

        n_samples = X.shape[0]
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)
        one_hot_y = np.eye(n_classes)[np.searchsorted(self.classes_, y)]

        # compute prior probabilities and initial logits
        prior = np.clip(np.mean(one_hot_y, axis=0), 1e-5, 1 - 1e-5)
        self.initial_logit_ = np.log(prior)

        # initialize F matrix with initial logits for all samples
        F = np.tile(self.initial_logit_, (n_samples, 1))

        for m in range(self.n_estimators):
            # 1) LDA transformation
            if m == 0:
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, y)
            else:
                # compute probabilities and residuals
                p = softmax(F)
                resid = one_hot_y - p
                # select labels corresponding to highest residuals
                labels = np.argmax(resid, axis=1)
                lda = LinearDiscriminantAnalysis()
                X_lda = lda.fit_transform(X, labels)

            # store the LDA transformation
            self.lda_transforms_.append(lda)

            # 2) update residuals
            p = softmax(F)
            resid = one_hot_y - p

            # 3) train one regression tree per class
            estims_m = []
            for k in range(n_classes):
                reg = DecisionTreeRegressor(max_depth=self.max_depth)
                reg.fit(X_lda, resid[:, k])
                estims_m.append(reg)
                # update logits for class k
                F[:, k] += self.learning_rate * reg.predict(X_lda)

            self.estimators_.append(estims_m)

        return self

    # -------------------------- inference --------------------------
    def _decision_function(self, X):
        """Computes the raw decision scores (logits)."""
        n_samples = X.shape[0]
        F = np.tile(self.initial_logit_, (n_samples, 1))
        for lda, estims_m in zip(self.lda_transforms_, self.estimators_):
            X_lda = lda.transform(X)
            for k, reg in enumerate(estims_m):
                F[:, k] += self.learning_rate * reg.predict(X_lda)
        return F

    def predict_proba(self, X):
        """Returns the predicted class probabilities."""
        return softmax(self._decision_function(X))

    def predict(self, X):
        """Returns the predicted class labels."""
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ---------- cross-validation ----------
def cross_validate_model(X, y, model, cv=10):
    """Performs stratified k-fold cross-validation."""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    accs, f1s = [], []

    for fold_idx, (tr_idx, te_idx) in enumerate(skf.split(X, y), start=1):
        X_tr, X_te = X[tr_idx], X[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        # clone the model for each fold to avoid data leakage
        model_fold = clone(model)
        model_fold.fit(X_tr, y_tr)

        # predictions and metrics
        y_pred = model_fold.predict(X_te)
        accs.append(accuracy_score(y_te, y_pred))
        f1s.append(f1_score(y_te, y_pred, average="macro"))

        # ---- progress message ----
        print(f"Fold {fold_idx}/{cv} completed.")

    return (
        np.mean(accs), np.std(accs),
        np.mean(f1s), np.std(f1s)
    )


# ---------- usage example ----------
model = CustomMulticlassGradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=7
)

mean_acc, std_acc, mean_f1, std_f1 = cross_validate_model(X, y_encoded, model, cv=10)
print(f"Mean Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Mean F1-score: {mean_f1:.4f} ± {std_f1:.4f}")


Fold 1/10 completed.
Fold 2/10 completed.
Fold 3/10 completed.
Fold 4/10 completed.
Fold 5/10 completed.
Fold 6/10 completed.
Fold 7/10 completed.
Fold 8/10 completed.
Fold 9/10 completed.
Fold 10/10 completed.
Accuratezza media: 0.9782 ± 0.0044
F1-score medio  : 0.9793 ± 0.0042
